# 00. 환경 설정 및 공통 유틸리티

**Phase 1 — GRU 단독 베이스라인 구축의 환경 노트북**

## 이 노트북의 역할

Phase 1의 모든 노트북(`01_*` ~ `04_*`)이 공통으로 사용하는 환경 설정을 부트스트랩합니다.
다른 노트북의 첫 코드 셀에서 다음 한 줄로 일괄 호출합니다.

```python
%run ./00_setup_and_utils.ipynb
```

## 협업 구조

환경 부트스트랩 로직은 **`scripts/setup.py`** 에 정의되어 있습니다.
이 노트북은 다음 두 가지 역할만 수행합니다.

1. **`scripts.setup` 모듈 import 및 환경 적용** (코드 셀)
2. **각 환경 컴포넌트의 의미·근거 설명** (마크다운 셀)

함수의 진실원(single source of truth)은 [scripts/setup.py](scripts/setup.py) 입니다.

## Phase 1 — GRU 목적

Phase1_LSTM 에서 확인된 과적합 문제 (파라미터/샘플 비 과도)를 완화하기 위해
GRU 로 교체한다. GRU 는 LSTM 대비 파라미터 수 약 25% 감소 (4게이트 → 2게이트).
동일 하이퍼파라미터, 동일 Walk-Forward 설정으로 LSTM 과 공정 비교한다.

## 시드 고정의 한계

시드를 고정해도 다음 요인으로 결과가 미세하게 다를 수 있습니다.
- GPU CUDA 비결정적 연산 (cuDNN 알고리즘)
- 멀티스레드 BLAS (NumPy/PyTorch의 OpenMP)
- 운영체제·Python 버전 차이

본 Phase 1은 CPU 환경 + `torch.use_deterministic_algorithms(True)`로 가능한 결정적 동작을 추구합니다.
그래도 환경 차이가 있을 수 있으므로 **메트릭은 소수점 4자리까지**만 비교합니다.

## §0. `scripts/` 모듈 import 경로 등록

이 노트북은 `Phase1_GRU/` 디렉토리에서 실행되므로, 같은 위치의 `scripts/` 패키지를
import 가능하게 하려면 `sys.path` 에 현재 작업 디렉토리를 추가합니다.

이미 추가되어 있으면 중복하지 않습니다. `%run` 으로 호출 시 다른 노트북에서도
이 sys.path 변경이 그대로 전파되어 `from scripts.X import Y` 가 그대로 작동합니다.

In [1]:
import sys
from pathlib import Path

_NB_DIR = Path.cwd()
if str(_NB_DIR) not in sys.path:
    sys.path.insert(0, str(_NB_DIR))

# scripts.setup 의 모든 public 인터페이스를 현재 namespace 로 가져온다.
from scripts.setup import (
    setup_korean_font,
    fix_seed,
    apply_display_defaults,
    ensure_result_dirs,
    bootstrap,
    SEED,
    BASE_DIR,
    RESULTS_DIR,
    RAW_DATA_DIR,
    SETTING_A_DIR,
    SETTING_B_DIR,
)

print(f'[OK] scripts.setup import 완료 — BASE_DIR = {BASE_DIR}')

[OK] scripts.setup import 완료 — BASE_DIR = /Users/yoonseokim/2025_main_bootcamp/4th_final_project/finance_project/시계열_Test/Phase1_GRU


## §1. 한글 폰트 설정

matplotlib 시각화에서 한글 레이블·제목이 □□□으로 깨지는 것을 방지합니다.

- **Windows**: 시스템 기본 한글 폰트인 `Malgun Gothic` 사용
- **macOS**: `AppleGothic`
- **Linux**: `koreanize_matplotlib` 패키지 (없으면 설치 안내)

또한 `axes.unicode_minus = False`로 설정해 마이너스 기호(−)가 박스로 깨지는 현상을 막습니다.

구현은 [scripts/setup.py](scripts/setup.py) 의 `setup_korean_font()` 함수에 있습니다.

In [2]:
_used_font = setup_korean_font()
print(f'[OK] 한글 폰트 설정 완료: {_used_font}')

[OK] 한글 폰트 설정 완료: AppleGothic


## §2. 시드 고정 (재현성 확보)

Python `random`, NumPy, PyTorch의 난수 생성기를 동일한 시드(42)로 초기화합니다.
또한 `torch.use_deterministic_algorithms(True)`로 PyTorch의 비결정적 알고리즘 사용을 차단합니다.

구현은 [scripts/setup.py](scripts/setup.py) 의 `fix_seed()` 함수에 있습니다.

In [3]:
import torch  # 버전 출력용

fix_seed(SEED)
print(f'[OK] 시드 고정 완료: SEED={SEED}')
print(f'[정보] PyTorch 버전: {torch.__version__}, CUDA 가용: {torch.cuda.is_available()}')

[OK] 시드 고정 완료: SEED=42
[정보] PyTorch 버전: 2.11.0, CUDA 가용: False


## §3. 경로 상수

결과물 저장 경로를 한 곳에 모아 정의합니다. 다른 노트북에서 이 변수들을 직접 참조하면
경로 변경 시 한 곳만 고쳐도 됩니다.

**구조**
```
Phase1_GRU/                ← BASE_DIR
├── 00_setup_and_utils.ipynb
├── 01_~ 04_*.ipynb
├── scripts/
│   └── setup.py            ← 이 BASE_DIR을 __file__ 기반으로 산출
└── results/                ← RESULTS_DIR
    ├── raw_data/           ← RAW_DATA_DIR
    ├── setting_A/          ← SETTING_A_DIR
    └── setting_B/          ← SETTING_B_DIR
```

**왜 `Path(__file__).resolve().parent.parent` 인가**: cwd 기반은 호출 위치에 따라 달라지지만,
`__file__` 기반은 `scripts/setup.py` 의 부모(`scripts/`)의 부모(`Phase1_GRU/`)로 항상 안정적으로 계산됩니다.

In [4]:
# 결과 디렉토리가 없으면 생성 (있으면 무시)
ensure_result_dirs()

print(f'[OK] 경로 상수 확인')
print(f'  BASE_DIR      = {BASE_DIR}')
print(f'  RAW_DATA_DIR  = {RAW_DATA_DIR}')
print(f'  SETTING_A_DIR = {SETTING_A_DIR}')
print(f'  SETTING_B_DIR = {SETTING_B_DIR}')

[OK] 경로 상수 확인
  BASE_DIR      = /Users/yoonseokim/2025_main_bootcamp/4th_final_project/finance_project/시계열_Test/Phase1_GRU
  RAW_DATA_DIR  = /Users/yoonseokim/2025_main_bootcamp/4th_final_project/finance_project/시계열_Test/Phase1_GRU/results/raw_data
  SETTING_A_DIR = /Users/yoonseokim/2025_main_bootcamp/4th_final_project/finance_project/시계열_Test/Phase1_GRU/results/setting_A
  SETTING_B_DIR = /Users/yoonseokim/2025_main_bootcamp/4th_final_project/finance_project/시계열_Test/Phase1_GRU/results/setting_B


## §4. 공통 import 및 표준 표시 옵션

데이터·모델 노트북에서 반복적으로 사용하는 라이브러리를 미리 import합니다.
`%run`으로 호출되면 이 변수들이 자동으로 호출 노트북의 namespace에 들어갑니다.

표시 옵션 적용은 [scripts/setup.py](scripts/setup.py) 의 `apply_display_defaults()` 에 위임합니다.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional

apply_display_defaults()

print('[OK] 공통 import + 표시 옵션 적용 완료')
print(f'  pandas {pd.__version__}, numpy {np.__version__}')

[OK] 공통 import + 표시 옵션 적용 완료
  pandas 2.3.3, numpy 2.4.4


## §5. 노트북 정상 로드 검증

환경 노트북이 정상적으로 로드되었는지 한눈에 확인합니다.
다른 노트북에서 `%run` 한 후 이 출력이 보이지 않으면 환경 설정에 문제가 있다는 신호입니다.

In [6]:
print('=' * 60)
print('  Phase 1 — GRU 단독 베이스라인 / 환경 설정 완료')
print('=' * 60)
print(f'  한글 폰트  : {_used_font}')
print(f'  시드       : {SEED}')
print(f'  결과 경로  : {RESULTS_DIR}')
print('  진실원     : scripts/setup.py')
print('=' * 60)

  Phase 1 — GRU 단독 베이스라인 / 환경 설정 완료
  한글 폰트  : AppleGothic
  시드       : 42
  결과 경로  : /Users/yoonseokim/2025_main_bootcamp/4th_final_project/finance_project/시계열_Test/Phase1_GRU/results
  진실원     : scripts/setup.py
